# LLMPirate Student Lab: IP Piracy using LLMs

This notebook is a **hands-on workshop** where you'll implement and compare different
strategies for using LLMs to perform IP piracy — modifying circuit netlists to evade
text-based similarity detection while preserving functionality.

## Lab Overview

You'll complete **5 tasks** that explore different approaches:

### Task 1: Direct Verilog Rewriting
Prompt the LLM to rewrite the entire Verilog RTL representation

### Task 2: Boolean Format Rewriting
Use Boolean equations (easier format) for circuit rewriting

### Task 3: Divide & Conquer Mapping (No Feedback)
Use the divide & conquer approach: get LLM mappings for individual gate types,
then apply to full circuit (single attempt, no retries)

### Task 4: Divide & Conquer with Iterative Feedback
Add a feedback loop to Task 3: retry failed mappings with error messages

### Task 5: Strategy Comparison
Compare all strategies across multiple runs and LLM models (GPT-3.5 vs GPT-4)

**Your Goal**: Determine which strategy is most effective at evading the **SIM** text
similarity detector (threshold 0.3) while maintaining functional correctness.

## Evaluation Metrics
- **SIM Score**: Text similarity (0-1, lower is better for evasion)
- **Functional Correctness**: Does the modified circuit work the same?
- **Success Rate**: How often does each strategy succeed?
- **Efficiency**: How many LLM queries are needed?

## Setup: Environment and Circuit Loading

First, let's set up the environment and load a test circuit for our experiments.

In [ ]:
#@title 0.1 Clone Repository & Install Dependencies
import os

# --- Set the data directory (adjust if running locally) ---
DATA_DIR = '/content/LLM_IP_assignment'   # <-- Colab default
# DATA_DIR = '.'   # <-- uncomment if running locally inside the repo

if not os.path.exists(DATA_DIR):
    !git clone https://github.com/matthewdelorenzo/LLM_IP_assignment.git {DATA_DIR}

os.chdir(DATA_DIR)
print('Working directory:', os.getcwd())

# Install Python dependencies
!pip install -q networkx openai

# Install Yosys (formal equivalence checker)
!apt-get install -y -q yosys

# Make the SIM binary executable
SIM_EXECUTABLE = os.path.join(DATA_DIR, 'sim', 'sim_text')
if os.path.exists(SIM_EXECUTABLE):
    os.chmod(SIM_EXECUTABLE, 0o755)
    print('SIM binary ready.')
else:
    print('WARNING: SIM binary not found at', SIM_EXECUTABLE)


Working directory: /content/LLM_IP_assignment
Reading package lists...
Building dependency tree...
Reading state information...
yosys is already the newest version (0.9-2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
SIM binary ready.


In [ ]:
#@title 0.2 Imports and Helper Functions
import sys, os, glob, pickle, copy, random, subprocess, time, re
import networkx as nx
import openai

# Add src/ directories to path
sys.path.insert(0, os.path.join(DATA_DIR, 'src'))
sys.path.insert(0, os.path.join(DATA_DIR, 'src_ablation_study_wo_SolB'))

# Core functions from original codebase
from characterize_circuit import (
    characterize_generic_netlist,
    create_LLM_circuit_query_template,
    is_gate_line,
    convert_circuit_to_Boolean_format,
    convert_circuit_from_Boolean_format,
)
from circuit_decomposer import (
    parse_expression,
    decompose_expression,
    final_decompose_expression,
)
from sim_evaluation import evaluate_sim_text
from config import DATASET_PATH, SIM_DIR

# Helper functions for the workshop
def evaluate_sim(orig_contents, pirated_contents, label='test'):
    """Save circuits to temp files and run SIM text comparison."""
    os.makedirs('tmp_eval', exist_ok=True)
    orig_path = f'tmp_eval/orig_{label}.v'
    pirated_path = f'tmp_eval/pirated_{label}.v'

    with open(orig_path, 'w') as f:
        f.write(orig_contents)
    with open(pirated_path, 'w') as f:
        f.write(pirated_contents)

    try:
        result = evaluate_sim_text(orig_path, pirated_path)
        if result.returncode != 0:
            print(f'SIM error (exit code {result.returncode}): {result.stderr}')
            return None
        if 'consists for' in result.stdout:
            score = float(
                result.stdout.split('\n')[-2]
                .split('consists for')[-1]
                .split('%')[0].strip()
            ) / 100.0
            return score
        return 0.0
    except Exception as e:
        print(f'SIM error: {e}')
        return None


def _strip_code_fences(text):
    """Remove markdown code fences from LLM responses."""
    if '```' in text:
        parts = text.split('```')
        if len(parts) >= 2:
            content = parts[1]
            lines = content.split('\n')
            if lines and lines[0].strip().lower() in ('', 'verilog', 'v', 'sv', 'systemverilog'):
                content = '\n'.join(lines[1:])
            return content.strip()
    return text


def _fix_verilog_declarations(verilog):
    """
    Fix common LLM Verilog output issues:
    - Add missing semicolons to input/output/wire/reg declaration lines
      that end a statement but are missing the terminating ';'
    """
    _DECL_KW = re.compile(r'^\s*(input|output|inout|wire|reg)\b')
    fixed_lines = []
    lines = verilog.splitlines()
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        if _DECL_KW.match(stripped):
            # Only add ';' if the line doesn't already end with ';' or ','
            # and it's not a continuation line (next line also starts a declaration
            # or is blank / endmodule)
            if stripped and not stripped.endswith(';') and not stripped.endswith(','):
                stripped = stripped + ';'
        fixed_lines.append(stripped)
    return '\n'.join(fixed_lines)


def _get_module_name(verilog):
    """Extract the top module name from a Verilog string."""
    for line in verilog.splitlines():
        m = re.match(r'\s*module\s+(\w+)', line)
        if m:
            return m.group(1)
    return None


def check_functional_equivalence(orig_verilog, transformed_verilog, label='test'):
    """
    Formal equivalence check using Yosys miter circuit + SAT solver.

    Runs the following Yosys flow:
        read_verilog <truth>
        read_verilog <generated>
        prep; proc; opt; memory;
        clk2fflogic;
        miter -equiv -flatten <gate> <gold> miter
        sat -seq 50 -verify -prove trigger 0 -show-all \
            -show-inputs -show-outputs -set-init-zero miter

    Returns: (is_equivalent: bool | None, details: str)
        True  — formally proven equivalent
        False — counterexample found (not equivalent)
        None  — Yosys error or inconclusive result
    """
    os.makedirs('tmp_eval', exist_ok=True)

    # Strip markdown fences and fix common LLM Verilog formatting issues
    orig_clean = _fix_verilog_declarations(_strip_code_fences(orig_verilog))
    trans_clean = _fix_verilog_declarations(_strip_code_fences(transformed_verilog))

    orig_module = _get_module_name(orig_clean)
    gen_module  = _get_module_name(trans_clean)

    if not orig_module:
        return None, "Could not extract module name from original Verilog"
    if not gen_module:
        return None, "Could not extract module name from transformed Verilog"

    # Rename both modules to unique names so Yosys never sees a name clash
    gold_name = 'gold_circuit'
    gate_name = 'gate_circuit'

    gold_verilog = re.sub(
        r'\bmodule\s+' + re.escape(orig_module) + r'\b',
        f'module {gold_name}', orig_clean, count=1
    )
    gate_verilog = re.sub(
        r'\bmodule\s+' + re.escape(gen_module) + r'\b',
        f'module {gate_name}', trans_clean, count=1
    )

    truth_path  = os.path.abspath(f'tmp_eval/truth_{label}.v')
    gen_path    = os.path.abspath(f'tmp_eval/gen_{label}.v')
    script_path = os.path.abspath(f'tmp_eval/eq_{label}.ys')

    with open(truth_path, 'w') as f:
        f.write(gold_verilog)
    with open(gen_path, 'w') as f:
        f.write(gate_verilog)

    yosys_script = (
        f"read_verilog {truth_path}\n"
        f"read_verilog {gen_path}\n"
        "prep; proc; opt; memory;\n"
        "clk2fflogic;\n"
        f"miter -equiv -flatten {gate_name} {gold_name} miter\n"
        "sat -seq 50 -verify -prove trigger 0 "
        "-show-all -show-inputs -show-outputs -set-init-zero miter\n"
    )
    with open(script_path, 'w') as f:
        f.write(yosys_script)

    print(f"🔍 Running Yosys equivalence check for '{label}'...")
    try:
        result = subprocess.run(
            ['yosys', '-s', script_path],
            capture_output=True, text=True, timeout=180
        )
        output = result.stdout + result.stderr

        if 'no model found: SUCCESS' in output:
            return True, "Circuits formally verified equivalent (Yosys miter+SAT)"
        elif 'proof did fail' in output:
            return False, "Circuits NOT equivalent — Yosys SAT found a counterexample"
        elif result.returncode != 0:
            err_lines = [l.strip() for l in output.splitlines() if 'ERROR' in l or 'Error' in l]
            err_msg = '; '.join(err_lines[:3]) if err_lines else output[-400:]
            return None, f"Yosys error: {err_msg}"
        else:
            return None, f"Yosys result inconclusive (exit {result.returncode})"

    except subprocess.TimeoutExpired:
        return None, "Yosys equivalence check timed out (180 s)"
    except FileNotFoundError:
        return None, "Yosys not found — run: !apt-get install -y yosys  (or re-run cell 3)"
    except Exception as e:
        return None, f"Equivalence check error: {e}"


def get_mapped_circuit(orig_file_contents, cached_circuit_mapping, mapping_strategy, rank=0):
    """
    Replace every gate in the original circuit with the LLM's cached rephrase.
    From evaluate_piracy_using_cached_mapping_multiprocessing_v2.py in the original codebase.
    """
    new_lines = []
    new_wires = []
    lines = orig_file_contents.split(';')
    lines = [line.strip() for line in lines if line.strip()]
    lines = [line.replace('\n', ' ') for line in lines]
    for line in lines:
        if line.startswith(('module ', 'input ', 'output ', 'wire ', 'endmodule')):
            new_lines.append(line)
            continue
        if is_gate_line(line):
            keep_line_same = False
            gate_type = line.split()[0] + '_' + str(line.count(','))
            if gate_type not in cached_circuit_mapping or len(cached_circuit_mapping[gate_type]) == 0:
                new_lines.append(line)
                continue
            if len(list(cached_circuit_mapping[gate_type].keys())) > 0:
                line = line.replace('(', ' ( ')
                line = line.replace(')', ' ) ')
                gate_type = line.split(' ')[0] + '_' + str(line.count(','))
                gate_name = line.split()[1].strip()
                op = line.split()[3].strip(',').strip()
                num_ips = line.count(',')
                ips = [line.split()[4+j].strip().strip(',').strip() for j in range(num_ips)]
                if mapping_strategy == 'random':
                    target_mapping_gate = random.choice(list(cached_circuit_mapping[gate_type].keys()))
                else:
                    if mapping_strategy.split('_')[0] == gate_type.split('_')[0].upper():
                        keep_line_same = True
                    elif mapping_strategy in list(cached_circuit_mapping[gate_type].keys()):
                        target_mapping_gate = copy.deepcopy(mapping_strategy)
                    else:
                        target_mapping_gate = random.choice(list(cached_circuit_mapping[gate_type].keys()))
                if keep_line_same:
                    new_lines.append(line)
                else:
                    mapped_circuit_str = cached_circuit_mapping[gate_type][target_mapping_gate][0]
                    nets = []
                    mapped_lines = mapped_circuit_str.split('\n')
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        nets.extend(tmp_inputs)
                        nets.append(tmp_output)
                    nets = list(set(nets))
                    inter_net_cnt = 0
                    for net in nets:
                        if net == 'Y':
                            for pat in ['Y = ', 'Y= ', 'Y =', 'Y=']:
                                if pat in mapped_circuit_str:
                                    mapped_circuit_str = mapped_circuit_str.replace(pat, op + pat[1:])
                                    break
                        elif net in ['A' + str(i) for i in range(1, 50)]:
                            idx = int(net[1:]) - 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                repl = pat.replace(net, ips[idx])
                                mapped_circuit_str = mapped_circuit_str.replace(pat, repl)
                        else:
                            wire_name = gate_name + '_inter_net_' + str(inter_net_cnt)
                            inter_net_cnt += 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                mapped_circuit_str = mapped_circuit_str.replace(pat, pat.replace(net, wire_name))
                            if net + ' = ' in mapped_circuit_str:
                                mapped_circuit_str = mapped_circuit_str.replace(net + ' = ', wire_name + ' = ')
                            new_wires.append(wire_name)
                    mapped_lines = mapped_circuit_str.split('\n')
                    cnt = 0
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        if not l:
                            continue
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        tmp_gate_type = l.split('=')[-1].split('(')[0].strip()
                        if tmp_gate_type in ['AND', 'OR', 'NAND', 'NOR'] and len(tmp_inputs) == 1:
                            tmp_inputs = [tmp_inputs[0], tmp_inputs[0]]
                        new_str = tmp_gate_type.lower() + ' ' + gate_name + '_' + str(cnt) + ' ( ' + tmp_output + ', ' + ', '.join(tmp_inputs) + ' )'
                        cnt += 1
                        new_lines.append(new_str)
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)
    if new_wires:
        for i in range(len(new_lines)):
            if new_lines[i].split()[0] == 'wire':
                new_lines[i] = new_lines[i] + ';\nwire ' + ', '.join(new_wires)
                break
    return ';\n'.join(new_lines)


def get_circuit_from_response(response):
    """Parse LLM response to extract gate equations (from original code)."""
    LLM_circuit = ""
    if "```" in response:
        lines = response.split("```")[1].split("\n")
        lines = [l for l in lines if l != '']
    elif "=" in response:
        lines = response.split("\n")
    else:
        return "Incorrect format"

    for line in lines:
        if ("=" in line) and (not line.startswith("wire ")):
            tmp_expression = line.split("=")[-1].strip().strip(";")
            if any(f"{g} " in tmp_expression for g in ["AND", "OR", "NAND", "NOR", "XOR", "XNOR"]):
                return "Incorrect format"
            tmp_op_net = line.split("=")[0].strip()
            decomposed = final_decompose_expression(tmp_expression, tmp_op_net)
            LLM_circuit += decomposed + "\n"

    if not LLM_circuit.strip():
        return "Incorrect format"
    return LLM_circuit.rstrip("\n")


SIM_THRESHOLD = 0.3

print('✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).')
print(f'SIM detection threshold: {SIM_THRESHOLD}')


✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).
SIM detection threshold: 0.3


In [ ]:
#@title 0.3 Load Circuit & Setup OpenAI Client

# Load the test circuit
DESIGN = 'C432'
VARIANT = 'c432-CS320'

circuit_path = os.path.join(DATASET_PATH, DESIGN, VARIANT, 'topModule.v')
with open(circuit_path, 'r') as f:
    orig_file_contents = f.read()

gate_counts = characterize_generic_netlist(orig_file_contents)
total_gates = sum(gate_counts.values())

print(f'Loaded circuit: {DESIGN}/{VARIANT}')
print(f'Total gates: {total_gates}')
print(f'Gate types: {dict(gate_counts)}')
print(f'Circuit size: {len(orig_file_contents)} characters')

# Show first 400 chars of the circuit
print(f'\nFirst 400 characters of circuit:')
print(orig_file_contents[:400] + '...')

# Set up OpenAI client for LLM experiments
# TODO: Set your OpenAI API key here
OPENAI_API_KEY = ""
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    openai.api_key = OPENAI_API_KEY
    print("\n✅ OpenAI API configured")
else:
    print("\n⚠️  WARNING: No API key set. You'll need this for live LLM querying.")
    print("Set OPENAI_API_KEY above or use the environment variable.")

Loaded circuit: C432/c432-CS320
Total gates: 212
Gate types: {'nand_2': 64, 'not_1': 60, 'xor_2': 32, 'nor_2': 19, 'xnor_2': 18, 'nand_4': 14, 'and_9': 3, 'and_8': 1, 'nand_3': 1}
Circuit size: 10789 characters

First 400 characters of circuit:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
        keyIn_0_10,...

✅ OpenAI API configured


---

## Student Task 1: Direct Verilog Rewriting

**Objective**: Prompt the LLM to rewrite the entire Verilog file using only specific gate types.

**Approach**: Send the full circuit to the LLM with instructions to use only NOR gates (or another target gate set).

**Challenges**:
- LLM must understand Verilog syntax
- Must handle the entire circuit at once (not divide & conquer)
- Must maintain correct wire connections
- No iterative feedback in basic version

**Your task**: Fill in the prompt template and run the experiment.

In [ ]:
#@title Task 1: Direct Verilog Rewriting Implementation

# TODO: Students can modify these settings
TASK1_TARGET_GATES = "NAND"  # Try: "NOR", "NAND", "AND and NOT"
TASK1_MODEL = "gpt-3.5-turbo-16k"  # Try: "gpt-3.5-turbo-16k"

# TODO: Students should improve this prompt template
task1_prompt = f"""
Rewrite the following Verilog circuit into a functionally equivalent Verilog circuit.

Strict requirements:
1. Preserve the exact same module name, input ports, output ports, and port order.
2. Preserve the exact same Boolean functionality for every output.
3. Use only {TASK1_TARGET_GATES} gates where possible.
4. You may introduce new internal wires, but do not rename existing inputs or outputs.
5. Do not remove or change any primary input or primary output.
6. Return only syntactically valid Verilog code.
7. Do not include markdown, comments, explanations, or code fences.
8. Every declaration must end with a semicolon.
9. Every gate instance must use valid Verilog gate syntax.

Original Verilog:
{orig_file_contents}
"""

print("Task 1 Prompt Summary:")
print("=" * 50)
print(f"Target gates: {TASK1_TARGET_GATES}")
print(f"Model: {TASK1_MODEL}")
print(f"Prompt length: {len(task1_prompt)} characters")
print("\nFirst 400 chars of prompt:")
print(task1_prompt[:400] + "...")

# Initialize result variables
task1_result = None
task1_sim_score = None
task1_evades = False
task1_functionally_correct = None

# Task 1 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 1...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK1_MODEL,
            messages=[{"role": "user", "content": task1_prompt}],
            temperature=0.0,
            max_tokens=4000
        )
        task1_result = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(task1_result)} chars)")
        print("First 400 chars of response:")
        print(task1_result[:400] + "...")

        # Evaluate functional correctness
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task1_result, "task1"
        )
        task1_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task1_sim_score = evaluate_sim(orig_file_contents, task1_result, "task1")
        task1_evades = task1_sim_score is not None and task1_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 1 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task1_functionally_correct else '❌ NO' if task1_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task1_sim_score:.4f}" if task1_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task1_evades else '❌ NO'}")
        print(f"Model Used: {TASK1_MODEL}")
        print(f"Target Gates: {TASK1_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task1_functionally_correct and task1_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 1 failed: {e}")
        task1_result = None
        task1_sim_score = None
        task1_evades = False
        task1_functionally_correct = None
else:
    print("⚠️ Skipping Task 1 execution (no API key)")

Task 1 Prompt Summary:
Target gates: NAND
Model: gpt-3.5-turbo-16k
Prompt length: 11482 characters

First 400 chars of prompt:

Rewrite the following Verilog circuit into a functionally equivalent Verilog circuit.

Strict requirements:
1. Preserve the exact same module name, input ports, output ports, and port order.
2. Preserve the exact same Boolean functionality for every output.
3. Use only NAND gates where possible.
4. You may introduce new internal wires, but do not rename existing inputs or outputs.
5. Do not remov...

EXECUTING TASK 1...
✅ LLM Response received (8481 chars)
First 400 chars of response:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
 

### Task 1 Analysis Questions (20 points)

**Record your results and answer these questions:**

***Note: It is ok to have results that are not functional and/or dont evade the piracy detectors! Analyze your findings as described below.***

1. **Success Rate**: Did the LLM successfully rewrite the circuit? If not, what went wrong?
  The LLM did not successfully rewrite the circuit. Although it generated a response that structurally resembled a Verilog module, the output contained syntax errors (unexpected end of file), which prevented the circuit from being parsed and verified. As a result, functional correctness could not be established.

2. **SIM Evasion**: What was your SIM score? Did it evade detection (< 0.3)?
  The SIM score was 0.2200, which is below the threshold of 0.3. Therefore, the rewritten circuit successfully evaded the similarity detector. This indicates that the LLM significantly altered the structure and representation of the circuit.

3. **Functional Correctness**: How would you verify the rewritten circuit works correctly? (Hint: you'd need to simulate both)
  Functional correctness could not be verified due to a syntax error in the generated Verilog code. In practice, correctness would be verified by simulating both the original and rewritten circuits or using a formal equivalence checker like Yosys. In this case, the equivalence check failed at the parsing stage, indicating that the generated circuit was not even syntactically valid.

4. **Prompt Engineering**: How might you improve the prompt to get better results?
  The prompt included strict constraints such as preserving inputs/outputs, maintaining exact functionality, and enforcing valid Verilog syntax. However, the LLM still failed to generate syntactically correct output. This suggests that direct full-circuit rewriting is too complex for the model, and that breaking the problem into smaller subproblems (as in later tasks) is likely more effective.

5. **Model Comparison**: Try both GPT-3.5 and GPT-4. Which performs better?
The experiment was run with both GPT-3.5-turbo-16k and GPT-4. GPT-3.5-turbo-16k was used as the primary model and produced a SIM score of 0.2200 but failed with syntax errors. GPT-4 was also tested and showed slightly better structural coherence in its output, but still failed to produce a syntactically valid full-circuit rewrite, confirming that the task complexity exceeds what either model can reliably handle in a single prompt.

**Your observations:**
```
Model used:
gpt-3.5-turbo-16k

SIM Score:
0.2200

Success/Failure:
Failure (Did not achieve overall success due to lack of functional correctness)

Key insights:
- The model was able to significantly reduce similarity (SIM < 0.3), successfully evading the SIM detector.
- However, the generated Verilog code contained syntax errors, which prevented functional verification.
- This shows that while LLMs can rewrite circuits to look different, they struggle to produce syntactically valid and functionally correct full-circuit outputs.
- Direct Verilog rewriting is too complex for the model, as it requires maintaining both structural integrity and logical equivalence simultaneously.
- Formal verification tools like Yosys are essential, as low similarity alone does not guarantee a valid or usable circuit.
```

---

## Student Task 2: Boolean Format Rewriting

**Objective**: Use Boolean equation format to make the task easier for the LLM.

**Key Insight**: Instead of raw Verilog, convert the circuit to Boolean equations:
- Verilog: `nand NAND2_1 ( N154, N118, N4 );`  
- Boolean: `N154 = NAND(N118, N4)`

This format is easier for LLMs to understand and manipulate.

**Your task**: Convert the circuit to Boolean format, get LLM to rephrase it, then convert back.

In [ ]:
#@title Task 2: Boolean Format Rewriting Implementation

# Convert circuit to Boolean format (this makes it easier for LLM)
circ_in_Boolean_format, remaining_orig_lines = convert_circuit_to_Boolean_format(orig_file_contents)

print("Boolean Format Conversion:")
print("=" * 30)
print(f"Boolean equations: {len([l for l in circ_in_Boolean_format.splitlines() if l.strip()])} lines")
print(f"Remaining structural lines: {len(remaining_orig_lines.splitlines())} lines")
print("\nFirst 300 chars of Boolean format:")
print(circ_in_Boolean_format[:300] + "...")

# TODO: Students configure task 2
TASK2_TARGET_GATES = "NAND"  # Try: "NOR", "NAND", "AND and NOT"
TASK2_MODEL = "gpt-3.5-turbo-16k"

# TODO: Students should improve this prompt
task2_prompt = f"""
Rewrite the following Boolean equations into functionally equivalent Boolean equations.

Strict requirements:
1. Preserve the exact same final output variables.
2. Preserve the exact same Boolean functionality.
3. Use only {TASK2_TARGET_GATES} operators.
4. Do not rename any original input variable.
5. Do not rename any original output variable.
6. You may introduce intermediate variables named temp_0, temp_1, temp_2, etc.
7. Each line must follow exactly this format:
   variable = GATE(input1, input2)
8. Do not use nested gates inside another gate.
   Bad: Y = NOR(NOR(A,B), C)
   Good:
   temp_0 = NOR(A, B)
   Y = NOR(temp_0, C)
9. Do not include markdown, explanations, numbering, or code fences.
10. Output only Boolean equations.

Boolean equations:
{circ_in_Boolean_format}
"""

print(f"\nTask 2 Configuration:")
print(f"Target gates: {TASK2_TARGET_GATES}")
print(f"Model: {TASK2_MODEL}")
print(f"Prompt length: {len(task2_prompt)} characters")

# Initialize results
task2_result = None
task2_sim_score = None
task2_evades = False
task2_functionally_correct = None

# Task 2 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 2...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK2_MODEL,
            messages=[{"role": "user", "content": task2_prompt}],
            temperature=0.0,
            max_tokens=4000
        )
        llm_boolean_response = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(llm_boolean_response)} chars)")
        print("First 300 chars of Boolean response:")
        print(llm_boolean_response[:300] + "...")

        # Convert back to Verilog format
        print("\n🔄 Converting back to Verilog format...")
        task2_result = convert_circuit_from_Boolean_format(llm_boolean_response, remaining_orig_lines)

        print(f"✅ Converted back to Verilog ({len(task2_result)} chars)")

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task2_result, "task2"
        )
        task2_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task2_sim_score = evaluate_sim(orig_file_contents, task2_result, "task2")
        task2_evades = task2_sim_score is not None and task2_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 2 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task2_functionally_correct else '❌ NO' if task2_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task2_sim_score:.4f}" if task2_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task2_evades else '❌ NO'}")
        print(f"Model Used: {TASK2_MODEL}")
        print(f"Target Gates: {TASK2_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task2_functionally_correct and task2_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 2 failed: {e}")
        task2_result = None
        task2_sim_score = None
        task2_evades = False
        task2_functionally_correct = None
else:
    print("⚠️ Skipping Task 2 execution (no API key)")

Boolean Format Conversion:
Boolean equations: 212 lines
Remaining structural lines: 13 lines

First 300 chars of Boolean format:
KeyWire_0[0] = NOT(N1)
N118 = XNOR(keyIn_0_0, KeyWire_0[0])
N119 = NOT(N4)
KeyWire_0[1] = NOT(N11)
N122 = XNOR(keyIn_0_1, KeyWire_0[1])
N123 = NOT(N17)
KeyWire_0[2] = NOT(N24)
KeyNOTWire_0[0] = XNOR(keyIn_0_2, KeyWire_0[2])
N126 = NOT(KeyNOTWire_0[0])
N127 = NOT(N30)
KeyWire_0[3] = NOT(N37)
N130 = X...

Task 2 Configuration:
Target gates: NAND
Model: gpt-3.5-turbo-16k
Prompt length: 6818 characters

EXECUTING TASK 2...
✅ LLM Response received (35 chars)
First 300 chars of Boolean response:
N433 = NAND(N381, N430, N431, N432)...

🔄 Converting back to Verilog format...
✅ Converted back to Verilog (2265 chars)

🔍 Checking functional equivalence...
🔍 Running Yosys equivalence check for 'task2'...
Functional check: Yosys error: /content/LLM_IP_assignment/tmp_eval/gen_task2.v:2: ERROR: syntax error, unexpected TOK_INPUT, expecting ';'

🔍 Evaluating with SIM detecto

### Task 2 Analysis Questions (20 points)

**Record your results and answer these questions:**

***Note: It is ok to have results that are not functional and/or dont evade the piracy detectors! Analyze your findings as described below.***

1. **Success Rate**: Did the LLM successfully rewrite the circuit? If not, what went wrong?
  The LLM did not successfully rewrite the circuit. It returned only a very short Boolean response, and after conversion back to Verilog, Yosys produced a syntax error. Therefore, the rewritten circuit could not be verified as functionally correct.

2. **SIM Evasion**: What was your SIM score? Did it evade detection (< 0.3)?
  The SIM score was 0.2200, which is below the detection threshold of 0.3. So the rewritten output did evade SIM detection.

3. **Functional Correctness**: What results did you get from the functionality check? What errors did you see, and why?
  Functional correctness was unknown because Yosys could not parse the generated Verilog. The error was a syntax error near line 2 of the generated file. This likely happened because the LLM response was incomplete or not compatible with the Boolean-to-Verilog conversion step.

4. **Prompt Engineering**: How did the prompt compare to Task 1 (length, content, etc)?
  Compared to Task 1, this prompt was more structured because it used Boolean equations instead of full Verilog. It also gave strict formatting rules and asked the model not to use nested gates. However, the model still failed to return a complete parseable Boolean rewrite.

5. **Model Comparison**: Try both GPT-3.5 and GPT-4. Which performs better?
  Both GPT-3.5-turbo-16k and GPT-4 were tested. GPT-3.5 was used as the primary model producing a SIM score of 0.2200 but with an incomplete Boolean response. GPT-4 was also evaluated but encountered list index errors during Boolean parsing, indicating inconsistency in its output format. Neither model succeeded in producing a functionally correct rewrite, suggesting the Boolean conversion pipeline is sensitive to output format regardless of model capability.

**Your observations:**
```
Model used:
gpt-3.5-turbo-16k

SIM Score:
0.2200

Success/Failure:
Failure. The circuit evaded SIM detection, but functional correctness was unknown because the converted Verilog produced a Yosys syntax error.

Key insights:
- Boolean format made the prompt simpler than raw Verilog, but the LLM output was incomplete.
- The SIM score was low enough to evade detection, but this does not imply correctness.
- The generated Boolean response caused syntax errors after conversion back to Verilog.
- Functional verification is essential because a low SIM score alone is not enough.
- Task 2 shows that intermediate Boolean formats help readability but still require strict parsing and validation.
```

---

## Student Task 3: Divide & Conquer Mapping (No Feedback)

**Objective**: Use the divide & conquer approach from the original LLMPirate paper.

**Approach**:
1. Identify all gate types in the circuit (nand_2, and_2, or_2, etc.)
2. For each gate type, ask LLM to rephrase it using target gates
3. Apply the gate mappings to the full circuit
4. No feedback loop - single attempt per gate type

**Advantages**: Simpler prompts, focused on individual gates
**Disadvantages**: No retry mechanism if LLM makes mistakes

In [ ]:
#@title Task 3: Divide & Conquer (No Feedback) Implementation

# Characterize the circuit to get all gate types and query templates
# (This is the "divide" step: one LLM query per gate type)
gate_counts = characterize_generic_netlist(orig_file_contents)
query_templates = create_LLM_circuit_query_template(gate_counts)

# Gates skipped by design: trivial (NOT/BUF) need no rephrasing
SKIP_GATE_TYPES = ['not_1', 'buf_1']
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates    = [g for g in gate_counts if g in SKIP_GATE_TYPES]

print("Gate types found:", list(gate_counts.keys()))
print(f"Non-trivial gate types to map: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial, no rephrasing needed): {skipped_gates}")

# TODO: Students configure task 3
TASK3_TARGET_GATES = ["OR", "NOT"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK3_MODEL = "gpt-4"

# TODO: Students should improve this prompt (from original GPT_communication_script.py)
def create_task3_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
    else:
        gate_rule = f"Use only {' and/or '.join(target_gates)} Boolean operators."

    return (
        "Rewrite the following single Boolean gate expression into an exactly equivalent form.\n\n"
        "Strict requirements:\n"
        f"1. {gate_rule}\n"
        "2. Preserve the exact truth table of the original gate.\n"
        "3. Use the same input names: A1, A2, A3, etc.\n"
        "4. Use the same output name: Y.\n"
        "5. You may introduce intermediate variables named T1, T2, T3, etc.\n"
        "6. Each line must be in exactly this format:\n"
        "   <output> = <gate type>(<input1>, <input2>)\n"
        "7. Do not use nested expressions.\n"
        "8. Do not include explanations, markdown, bullets, or code fences.\n"
        "9. Output only the rewritten equations.\n\n"
        "Original expression:\n"
        + template
    )

print(f"\nTask 3 Configuration:")
print(f"Target gates: {TASK3_TARGET_GATES}")
print(f"Model: {TASK3_MODEL}")

# Initialize results
task3_circuit_mapping = {}
task3_final_circuit = None
task3_sim_score = None
task3_evades = False
task3_functionally_correct = None

# Task 3 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 3...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK3_TARGET_GATES)

        # Initialize all gate types with empty mappings (unmapped gates stay as-is)
        for gate_type in gate_counts:
            task3_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, ask the LLM for a single-shot rephrasing
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            prompt = create_task3_prompt(template, TASK3_TARGET_GATES)

            response = openai.chat.completions.create(
                model=TASK3_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=500
            )

            llm_response = response.choices[0].message.content
            LLM_circuit = get_circuit_from_response(llm_response)

            if "Incorrect format" in LLM_circuit:
                print(f"  ⚠️ Could not parse LLM response — keeping original gate")
                continue

            print(f"  ✅ Mapping: {LLM_circuit.strip()}")
            task3_circuit_mapping[gate_type][target_key] = [LLM_circuit]

        mapped_count = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")

        # Apply all mappings to the full circuit ("conquer" step)
        print("\n🔄 Applying gate mappings to full circuit...")
        task3_final_circuit = get_mapped_circuit(orig_file_contents, task3_circuit_mapping, target_key)

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task3_final_circuit, "task3"
        )
        task3_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task3_sim_score = evaluate_sim(orig_file_contents, task3_final_circuit, "task3")
        task3_evades = task3_sim_score is not None and task3_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 3 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Functional Correctness: {'✅ YES' if task3_functionally_correct else '❌ NO' if task3_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task3_sim_score:.4f}" if task3_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task3_evades else '❌ NO'}")
        print(f"Model Used: {TASK3_MODEL}")
        print(f"Target Gates: {TASK3_TARGET_GATES}")
        overall_success = task3_functionally_correct and task3_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 3 failed: {e}")
        task3_circuit_mapping = {}
        task3_final_circuit = None
        task3_sim_score = None
        task3_evades = False
        task3_functionally_correct = None
else:
    print("⚠️ Skipping Task 3 execution (no API key)")


Gate types found: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Non-trivial gate types to map: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial, no rephrasing needed): ['not_1']

Task 3 Configuration:
Target gates: ['OR', 'NOT']
Model: gpt-4

EXECUTING TASK 3...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  ✅ Mapping: T1 = OR(A1, A2)
Y = NOT(T1)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  ✅ Mapping: T1 = NOT(A1)
T2 = NOT(A2)
T3 = OR(A1, T2)
T4 = OR(T1, A2)
Y = OR(T3, T4)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  ✅ Mapping: T1 = NOT(A1)
T2 = NOT(A2)
Y = OR(T1, T2)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  ✅ Mapping: T1 = NOT(A1)
T2 = NOT(A2)
T3 = OR(A1, T2)
T4 = OR(A2, T1)
Y = OR(T3, T4)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  ✅ Mapping: T1 = OR(A1, A2)
T2 = OR(T1, A3)
T3 = OR(T2, A4)
Y = NOT(T3)

--- Gate type: and_9  

### Task 3 Analysis Questions (60)

**Record your results and answer these questions:**

***Note: It is ok to have results that are not functional and/or dont evade the piracy detectors! Analyze your findings as described below.***

1. **Success Rate**: Did the LLM successfully rewrite the circuit? If not, what went wrong?
  The LLM successfully generated mappings for all 8/8 non-trivial gate types, while not_1 was skipped as a trivial gate. However, the final rewritten circuit was not functionally correct after applying the mappings to the full circuit.

2. **SIM Evasion**: What was your SIM score? Did it evade detection (< 0.3)?
  The SIM score was 0.2500, which is below the detection threshold of 0.3. Therefore, the rewritten circuit successfully evaded the SIM detector.

3. **Functional Correctness**: What results did you get from the functionality check? What errors did you see, and why?
  Functional correctness failed. Yosys reported that the circuits were not equivalent and found a counterexample. This means at least one input combination produced different outputs between the original and rewritten circuits. The likely cause is that one or more gate mappings were logically incorrect or became incorrect when applied throughout the full circuit.

4. **Prompt Engineering**: How does the strategy compare to Task 2? (Hint: how is the circuit constructed compared to Task 2)
  Compared to Task 2, Task 3 used a divide-and-conquer strategy. Instead of asking the LLM to rewrite the entire Boolean circuit, it asked the model to rewrite individual gate types and then applied those mappings to the full circuit. This made the prompts shorter and easier for the model to answer, but one incorrect mapping could still break the final circuit.

5. **Model Comparison**: Try both GPT-3.5 and GPT-4. Which performs better?
  This run used GPT-4, which generated all required gate mappings successfully. Compared to GPT-3.5, GPT-4 is generally better at following structured prompts and producing complete mappings. However, even GPT-4 failed to preserve full-circuit functional correctness.

**Your observations:**
```
Model used:
gpt-4

SIM Score:
0.2500

Success/Failure:
Failure. The circuit evaded SIM detection, but it failed functional equivalence.

Key insights:
- The divide-and-conquer approach successfully mapped all 8/8 non-trivial gate types.
- Using OR and NOT gates improved the SIM score below the 0.3 threshold.
- Even though the mappings were generated successfully, the full circuit was not functionally equivalent.
- Yosys found a counterexample, meaning the rewritten circuit behaves differently for at least one input case.
- This shows that local gate-level rewriting can reduce similarity, but it does not guarantee global circuit correctness.
```

---

## Student Task 4: Divide & Conquer with Iterative Feedback

**Objective**: Add a feedback loop to Task 3 to improve success rates.

**Approach**: Same as Task 3, but:
- If format parsing fails → send format feedback
- If functional verification fails → send functional feedback  
- Retry up to 5 times per gate type

**This matches the original LLMPirate pipeline with SolC (self-correction).**

In [ ]:
import itertools, re

def eval_gate(gate, args):
    gate = gate.upper()
    if gate == "AND":
        return int(all(args))
    if gate == "OR":
        return int(any(args))
    if gate == "NAND":
        return int(not all(args))
    if gate == "NOR":
        return int(not any(args))
    if gate == "XOR":
        return int(sum(args) % 2 == 1)
    if gate == "XNOR":
        return int(sum(args) % 2 == 0)
    if gate == "NOT":
        return int(not args[0])
    if gate == "BUF":
        return int(args[0])
    raise ValueError(f"Unknown gate: {gate}")

def parse_gate_line(line):
    line = line.strip().strip(";")
    m = re.match(r"(\w+)\s*=\s*(\w+)\((.*?)\)", line)
    if not m:
        return None
    out = m.group(1)
    gate = m.group(2)
    ins = [x.strip() for x in m.group(3).split(",") if x.strip()]
    return out, gate, ins

def evaluate_mapping(lines, values):
    env = dict(values)
    for line in lines.strip().splitlines():
        parsed = parse_gate_line(line)
        if parsed is None:
            return None
        out, gate, ins = parsed
        try:
            args = [env[x] for x in ins]
        except KeyError:
            return None
        env[out] = eval_gate(gate, args)
    return env.get("Y", None)

def check_gate_mapping_equivalence(template, llm_mapping):
    """
    Checks whether one LLM gate mapping is truth-table equivalent
    to the original gate template.
    """
    orig = parse_gate_line(template)
    if orig is None:
        return False, None, None, None

    _, orig_gate, orig_inputs = orig
    input_vars = orig_inputs

    for bits in itertools.product([0, 1], repeat=len(input_vars)):
        values = dict(zip(input_vars, bits))

        expected = eval_gate(orig_gate, [values[x] for x in orig_inputs])
        actual = evaluate_mapping(llm_mapping, values)

        if actual != expected:
            return False, values, expected, actual

    return True, None, None, None

In [ ]:
#@title Task 4: Divide & Conquer WITH Iterative Feedback Implementation

# Reuse gate characterization from Task 3
# (gate_counts, query_templates, SKIP_GATE_TYPES,
#  non_trivial_gates, skipped_gates are already set)
print(f"Gate types to process: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial): {skipped_gates}")

# TODO: Students configure task 4
TASK4_TARGET_GATES = ["OR", "NOT"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK4_MODEL = "gpt-4"  # Try: "gpt-4" for better reasoning
TASK4_MAX_TRIALS = 8  # Max feedback rounds per gate type (matches original source)

# TODO: Students should improve this prompt (from original GPT_communication_script.py)
def create_task4_prompt(template, target_gates):
    return (
        "Rewrite the following Boolean gate expression into an exactly equivalent form.\n\n"
        "Strict requirements:\n"
        f"1. Use only {target_gates[0]} gates.\n"
        "2. Preserve the exact truth table of the original expression.\n"
        "3. Use the same inputs A1, A2, A3, etc.\n"
        "4. Use output name Y exactly.\n"
        "5. You may introduce intermediate variables named T1, T2, T3, etc.\n"
        "6. Every gate must be a two-input gate.\n"
        "7. Do not use nested expressions.\n"
        "8. Return only equations in this exact format:\n"
        "   T1 = NAND(A1, A2)\n"
        "   Y = NAND(T1, T1)\n"
        "9. No markdown, no comments, no explanation.\n\n"
        "Original expression:\n"
        + template
    )

def create_task4_format_feedback(template, target_gates):
    return (
        "This is not the correct format. Can you try again in this format: "
        "<output> = <gate type> (<inputs>)? "
        f"Use only {', '.join(target_gates)} operator(s). "
        "Below is the original circuit:\n" + template
    )

def create_task4_functional_feedback(template, target_gates):
    return (
        "The previous mapping was not functionally equivalent.\n"
        "Retry using only two-input NAND gates.\n"
        "Preserve the exact truth table.\n"
        "Use the same input names and output Y.\n"
        "Return only equations, no explanation.\n\n"
        "Original expression:\n"
        + template
    )

print(f"\nTask 4 Configuration:")
print(f"Target gates: {TASK4_TARGET_GATES}")
print(f"Model: {TASK4_MODEL}")
print(f"Max trials per gate type: {TASK4_MAX_TRIALS}")

# Initialize results
task4_circuit_mapping = {}
task4_final_circuit = None
task4_sim_score = None
task4_evades = False
task4_functionally_correct = None
task4_gate_trial_counts = {}

# Task 4 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 4...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK4_TARGET_GATES)

        # Initialize all gate types with empty mappings
        for gate_type in gate_counts:
            task4_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, query LLM with iterative feedback on failure
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            messages = [{"role": "user", "content": create_task4_prompt(template, TASK4_TARGET_GATES)}]
            num_trials = 0
            success = False

            while num_trials < TASK4_MAX_TRIALS:
                num_trials += 1
                print(f"  Trial {num_trials}/{TASK4_MAX_TRIALS}")

                # Use full conversation history for context (feedback loop)
                response = openai.chat.completions.create(
                    model=TASK4_MODEL,
                    messages=messages,
                    temperature=0.0,
                    max_tokens=500
                )
                llm_response = response.choices[0].message.content
                messages.append({"role": "assistant", "content": llm_response})

                LLM_circuit = get_circuit_from_response(llm_response)

                if "Incorrect format" in LLM_circuit:
                    print(f"  ⚠️ Format error — sending feedback")
                    feedback = create_task4_format_feedback(template, TASK4_TARGET_GATES)
                    messages.append({"role": "user", "content": feedback})
                    continue

                # Local truth-table validation before accepting the mapping
                is_valid, counterexample, expected, actual = check_gate_mapping_equivalence(template, LLM_circuit)

                if not is_valid:
                    print(f"  ⚠️ Functional mismatch — retrying")
                    print(f"     Counterexample: {counterexample}, expected={expected}, actual={actual}")

                    feedback = (
                        "The previous mapping was not functionally equivalent.\n"
                        f"Counterexample input: {counterexample}\n"
                        f"Expected output Y = {expected}, but your mapping produced Y = {actual}.\n"
                        "Try again.\n"
                        f"Use only {', '.join(TASK4_TARGET_GATES)} gates.\n"
                        "Preserve the exact truth table.\n"
                        "Return only equations in this format:\n"
                        "<output> = <gate type>(<inputs>)\n\n"
                        "Original expression:\n"
                        + template
                    )

                    messages.append({"role": "user", "content": feedback})
                    continue

                print(f"  ✅ Mapping validated: {LLM_circuit.strip()}")
                task4_circuit_mapping[gate_type][target_key] = [LLM_circuit]
                success = True
                break

            task4_gate_trial_counts[gate_type] = num_trials
            if not success:
                print(f"  ❌ No valid mapping after {TASK4_MAX_TRIALS} trials — keeping original gate")

        mapped_count = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
        avg_trials = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")

        # Apply all mappings to the full circuit
        print("\n🔄 Applying gate mappings to full circuit...")
        task4_final_circuit = get_mapped_circuit(orig_file_contents, task4_circuit_mapping, target_key)

        # Check functional equivalence on the full mapped circuit
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task4_final_circuit, "task4"
        )
        task4_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task4_sim_score = evaluate_sim(orig_file_contents, task4_final_circuit, "task4")
        task4_evades = task4_sim_score is not None and task4_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 4 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")
        print(f"Functional Correctness: {'✅ YES' if task4_functionally_correct else '❌ NO' if task4_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task4_sim_score:.4f}" if task4_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task4_evades else '❌ NO'}")
        print(f"Model Used: {TASK4_MODEL}")
        print(f"Target Gates: {TASK4_TARGET_GATES}")
        overall_success = task4_functionally_correct and task4_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

        # Show per-gate-type trial counts
        print("\nPer-gate-type trial counts:")
        for gt in gate_counts:
            if gt in SKIP_GATE_TYPES:
                print(f"  {gt}: — (trivial gate, skipped by design)")
            elif gt in task4_gate_trial_counts:
                trials = task4_gate_trial_counts[gt]
                mapped = '✅' if task4_circuit_mapping.get(gt) else '❌'
                print(f"  {gt}: {trials} trial(s) {mapped}")

    except Exception as e:
        print(f"❌ Task 4 failed: {e}")
        task4_circuit_mapping = {}
        task4_final_circuit = None
        task4_sim_score = None
        task4_evades = False
        task4_functionally_correct = None
        task4_gate_trial_counts = {}
else:
    print("⚠️ Skipping Task 4 execution (no API key)")


Gate types to process: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial): ['not_1']

Task 4 Configuration:
Target gates: ['OR', 'NOT']
Model: gpt-4
Max trials per gate type: 8

EXECUTING TASK 4...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  Trial 1/8
  ⚠️ Functional mismatch — retrying
     Counterexample: {'A1': 0, 'A2': 0}, expected=1, actual=0
  Trial 2/8
  ⚠️ Functional mismatch — retrying
     Counterexample: {'A1': 0, 'A2': 0}, expected=1, actual=0
  Trial 3/8
  ✅ Mapping validated: T1 = NOT(A1)
T2 = NOT(A2)
Y = OR(T1, T2)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  Trial 1/8
  ⚠️ Functional mismatch — retrying
     Counterexample: {'A1': 1, 'A2': 1}, expected=0, actual=1
  Trial 2/8
  ⚠️ Functional mismatch — retrying
     Counterexample: {'A1': 0, 'A2': 0}, expected=0, actual=1
  Trial 3/8
  ⚠️ Functional mismatch — retrying
     Counterexample: {'A1': 0, 'A2': 0}, expected=0, actual=1
  Trial 4/8
  ✅ Mapp

### Task 4 Analysis Questions (20 pts)

**Record your results and answer these questions:**

***Note: It is ok to have results that are not functional and/or dont evade the piracy detectors! Analyze your findings as described below.***

1. **Success Rate**: Was the LLM able to determine functionally equivalent transformations for the circuit gates? Which target gates did you try, and which worked well?
  The LLM partially succeeded. It generated valid mappings for 7 out of 8 non-trivial gate types, but failed for one gate type (xnor_2) even after multiple retries. Despite the iterative feedback loop, the final circuit was still not functionally equivalent.

2. **SIM Evasion**: What was your SIM score? Did it evade detection (< 0.3)?
  The SIM score was 0.3300, which is above the threshold of 0.3. Therefore, the circuit did not evade detection.

3. **Functional Correctness**: What results did you get from the functionality check? What errors did you see, and why?
  Functional correctness failed. Yosys reported that the circuits were not equivalent and found a counterexample. This indicates that the rewritten circuit produces incorrect outputs for at least one input combination. The likely causes are:

* Incorrect mapping for one or more gate types (especially the failed xnor_2)
* Error propagation when applying mappings across the full circuit
* Local correctness not translating to global correctness

4. **Prompt Engineering**: How does the strategy compare to Task 3?
  Compared to Task 3, Task 4 introduced an iterative feedback loop with up to 8 retries per gate type. This improved mapping coverage slightly (though still incomplete), but did not fix correctness issues. The feedback loop helps correct formatting errors but struggles to fix deeper logical errors in gate transformations.

5. **Model Comparison**: Try both GPT-3.5 and GPT-4. Which performs better?
  This run used GPT-4, which handled multiple retries and produced most mappings successfully. Compared to GPT-3.5, GPT-4 is better at following iterative corrections, but still fails to ensure full functional correctness at the circuit level.

**Your observations:**
```
Model used:
gpt-4

SIM Score:
0.3300

Success/Failure:
Failure. The circuit was not functionally correct and did not evade SIM detection.

Key insights:
- Iterative feedback improved mapping attempts but did not guarantee correct mappings.
- Only 7/8 gate types were successfully mapped; failure on one gate type impacted the full circuit.
- The SIM score worsened compared to Task 3, indicating less effective obfuscation.
- Functional correctness remains the main challenge, even with multiple retries.
- The feedback loop helps with formatting but is not sufficient for ensuring logical equivalence.
- Errors in individual gate mappings propagate and break the full circuit behavior.
```

---

## Reference Baseline + Strategy Comparison (Task 5)

### Part A: Reference Baseline — Original Research Mappings

Before comparing, this cell first establishes a **reference baseline** using the pre-generated gate mappings from the original LLMPirate research. These cached mappings were created by the research team by running the full pipeline (with iterative feedback) across GPT-3.5, GPT-4, Claude, Gemini, Llama 3, and other models.

**Why this matters**: It lets you see how your generated mappings compare against research-grade mappings on the same circuit.

**Note**: The cache only covers gate types from the original benchmark suite. Gates like `and_8` / `and_9` (≥8 inputs) have no cached mapping, so those gates remain unchanged in the reference circuit.

### Part B: Strategy Comparison

Compare all strategies on the same circuit and metrics:
1. **Task 1**: Direct Verilog rewriting
2. **Task 2**: Boolean format rewriting
3. **Task 3**: Divide & conquer (no feedback)
4. **Task 4**: Divide & conquer (with feedback)
5. **Reference**: Original research team's cached mappings

**Metrics**: SIM evasion score, functional correctness, overall success


In [ ]:
#@title Task 5: Reference Baseline + Strategy Comparison

import pandas as pd
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# PART A: Reference Baseline — Original Research Cached Mappings
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("PART A: REFERENCE BASELINE")
print("=" * 60)

# TODO: Students can switch LLM to compare baselines
REFERENCE_LLM = 'GPT3dot5'   # Try: 'GPT3dot5', 'GPT4', 'Claude', 'Gemini', 'llama3'
REFERENCE_STRATEGY = 'random'  # 'random' picks best available per gate type

mapping_path = os.path.join(DATA_DIR, 'src', f'cached_circuit_mapping_{REFERENCE_LLM}.pkl')
with open(mapping_path, 'rb') as f:
    reference_mapping = pickle.load(f)

print(f"Loaded cached mapping: {REFERENCE_LLM}")
print(f"\nCoverage for this circuit's gate types:")
for gt in gate_counts:
    strategies = reference_mapping.get(gt, {})
    if strategies:
        print(f"  ✅ {gt}: {list(strategies.keys())}")
    else:
        print(f"  ⬜ {gt}: no cached mapping (gate kept as-is)")

print(f"\n🔄 Applying {REFERENCE_LLM} cached mappings (strategy: {REFERENCE_STRATEGY})...")
ref_circuit = get_mapped_circuit(orig_file_contents, reference_mapping, REFERENCE_STRATEGY)

print("\n🔍 Checking functional equivalence...")
ref_equiv, ref_details = check_functional_equivalence(orig_file_contents, ref_circuit, 'reference')
print(f"Functional check: {ref_details}")

print("\n🔍 Evaluating with SIM detector...")
ref_sim_score = evaluate_sim(orig_file_contents, ref_circuit, 'reference')
ref_evades = ref_sim_score is not None and ref_sim_score < SIM_THRESHOLD

print(f"\n📊 REFERENCE BASELINE RESULTS ({REFERENCE_LLM}):")
print(f"Functional Correctness: {'✅ YES' if ref_equiv else '❌ NO' if ref_equiv is False else '⚠️ UNKNOWN'}")
print(f"SIM Score: {ref_sim_score:.4f}" if ref_sim_score is not None else "SIM Score: Error")
print(f"Evades Detection: {'✅ YES' if ref_evades else '❌ NO'}")

# ─────────────────────────────────────────────────────────────
# PART B: Strategy Comparison
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART B: COMPREHENSIVE STRATEGY COMPARISON")
print("=" * 60)

all_results = []

# Task 1 Results
if 'task1_result' in locals() and task1_sim_score is not None:
    all_results.append({
        'Task': 'Task 1: Direct Verilog',
        'Method': 'Direct LLM rewriting',
        'SIM_Score': task1_sim_score,
        'Functional': task1_functionally_correct,
        'Overall_Success': bool(task1_functionally_correct and task1_evades),
        'Model': TASK1_MODEL,
        'Notes': 'Single-shot, full circuit'
    })

# Task 2 Results
if 'task2_result' in locals() and task2_sim_score is not None:
    all_results.append({
        'Task': 'Task 2: Boolean Format',
        'Method': 'Boolean intermediate format',
        'SIM_Score': task2_sim_score,
        'Functional': task2_functionally_correct,
        'Overall_Success': bool(task2_functionally_correct and task2_evades),
        'Model': TASK2_MODEL,
        'Notes': 'Via Boolean equations'
    })

# Task 3 Results
if 'task3_final_circuit' in locals() and task3_final_circuit is not None:
    mapped_count3 = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
    all_results.append({
        'Task': 'Task 3: D&C No Feedback',
        'Method': 'Gate-type mapping, single-shot',
        'SIM_Score': task3_sim_score,
        'Functional': task3_functionally_correct,
        'Overall_Success': bool(task3_functionally_correct and task3_evades),
        'Model': TASK3_MODEL,
        'Notes': f'{mapped_count3}/{len(non_trivial_gates)} gate types mapped'
    })

# Task 4 Results
if 'task4_final_circuit' in locals() and task4_final_circuit is not None:
    mapped_count4 = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
    avg_trials4 = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
    all_results.append({
        'Task': 'Task 4: D&C With Feedback',
        'Method': 'Gate-type mapping, iterative feedback',
        'SIM_Score': task4_sim_score,
        'Functional': task4_functionally_correct,
        'Overall_Success': bool(task4_functionally_correct and task4_evades),
        'Model': TASK4_MODEL,
        'Notes': f'{mapped_count4}/{len(non_trivial_gates)} mapped, {avg_trials4:.1f} avg trials'
    })

# Reference Baseline Results
covered_gates = sum(1 for gt in gate_counts if reference_mapping.get(gt))
all_results.append({
    'Task': f'Reference: {REFERENCE_LLM} Cache',
    'Method': 'Pre-validated research mappings',
    'SIM_Score': ref_sim_score,
    'Functional': ref_equiv,
    'Overall_Success': bool(ref_equiv and ref_evades),
    'Model': REFERENCE_LLM,
    'Notes': f'{covered_gates}/{len(gate_counts)} gate types in cache'
})

# ── Print comparison table ──────────────────────────────────
print("\n📋 SUMMARY TABLE:")
print("-" * 60)
for row in all_results:
    print(f"\n{row['Task']}:")
    print(f"  Method: {row['Method']}")
    print(f"  Model:  {row['Model']}")
    print(f"  SIM Score: {row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "  SIM Score: N/A")
    func = row['Functional']
    print(f"  Functional: {'✅ YES' if func else ('❌ NO' if func is False else '⚠️ UNKNOWN')}")
    print(f"  Overall Success: {'✅ YES' if row['Overall_Success'] else '❌ NO'}")
    print(f"  Notes: {row['Notes']}")

# ── Ranking ─────────────────────────────────────────────────
print(f"\n🎯 STRATEGY EFFECTIVENESS RANKING:")
print("-" * 40)

sorted_results = sorted(
    all_results,
    key=lambda r: (int(r['Overall_Success']), -(r['SIM_Score'] if r['SIM_Score'] is not None else 1.0))
)[::-1]

for rank, row in enumerate(sorted_results, 1):
    icon = '🏆' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '📍'
    sim_str = f"SIM={row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "SIM=N/A"
    success_str = '✅' if row['Overall_Success'] else '❌'
    print(f"{rank}. {icon} {row['Task']:35s}  {success_str}  {sim_str}")

# ── Key insights ─────────────────────────────────────────────
print(f"\n📈 KEY INSIGHTS:")
valid_sim = [r for r in all_results if r['SIM_Score'] is not None]
if valid_sim:
    best = min(valid_sim, key=lambda r: r['SIM_Score'])
    print(f"• Lowest SIM score: {best['SIM_Score']:.4f}  ({best['Task']})")
    if ref_sim_score is not None:
        for row in all_results:
            if row['Task'].startswith('Task') and row['SIM_Score'] is not None:
                diff = row['SIM_Score'] - ref_sim_score
                indicator = '🔺 worse' if diff > 0.02 else ('🔻 better' if diff < -0.02 else '≈ similar')
                print(f"• {row['Task']:35s} vs reference: {diff:+.4f} ({indicator})")

model_results = defaultdict(list)
for r in all_results:
    model_results[r['Model']].append(r['Overall_Success'])
if len(model_results) > 1:
    print(f"\n🤖 MODEL COMPARISON:")
    for model, successes in model_results.items():
        rate = sum(successes) / len(successes)
        print(f"• {model}: {rate:.0%} success rate")


PART A: REFERENCE BASELINE
Loaded cached mapping: GPT3dot5

Coverage for this circuit's gate types:
  ✅ nand_2: ['AND_NOT', 'OR_NOT']
  ⬜ not_1: no cached mapping (gate kept as-is)
  ✅ xor_2: ['NAND']
  ✅ nor_2: ['AND_NOT']
  ✅ xnor_2: ['NOR']
  ✅ nand_4: ['NOR', 'AND_NOT', 'OR_NOT']
  ⬜ and_9: no cached mapping (gate kept as-is)
  ⬜ and_8: no cached mapping (gate kept as-is)
  ✅ nand_3: ['AND_NOT', 'OR_NOT']

🔄 Applying GPT3dot5 cached mappings (strategy: random)...

🔍 Checking functional equivalence...
🔍 Running Yosys equivalence check for 'reference'...
Functional check: Circuits formally verified equivalent (Yosys miter+SAT)

🔍 Evaluating with SIM detector...

📊 REFERENCE BASELINE RESULTS (GPT3dot5):
Functional Correctness: ✅ YES
SIM Score: 0.2500
Evades Detection: ✅ YES

PART B: COMPREHENSIVE STRATEGY COMPARISON

📋 SUMMARY TABLE:
------------------------------------------------------------

Task 1: Direct Verilog:
  Method: Direct LLM rewriting
  Model:  gpt-3.5-turbo-16k
  SIM Sc

## Final Analysis Questions (20 pts)

**Based on your experiments above, summarize your results from your responses in Tasks 1-4. Some guiding questions are below to assist in your discussion.**

***(Not all questions are required, but aim for at least 1-2 sentences for each section below).***

### 1. Strategy Effectiveness
- Which strategy had the highest success rate?
- Which strategy had the best SIM evasion rate?
- What errors did you see most often from the LLM generated designs?

  The reference GPT3dot5 cached mapping had the highest success rate because it was the only method that achieved both functional correctness and SIM evasion. Among the live LLM strategies, Task 1 had the lowest SIM score at 0.2200, but it did not achieve functional correctness. The most common errors were syntax errors, incomplete outputs, and functional mismatches found by Yosys.

### 2. Model Comparison
- Did GPT-4 significantly outperform GPT-3.5?
  
  GPT-4 performed better than GPT-3.5 in generating structured gate mappings for Tasks 3 and 4, but it still did not achieve overall success. The cached GPT3dot5 reference mapping performed best because it used pre-validated mappings rather than a fresh single LLM response.

### 3. Target Gate Analysis
- Which target gate set (NOR vs NAND) worked better?
- Why do you think certain gates are easier for LLMs?

  The OR/NOT target gate set worked better than NAND/NOR in my later experiments for reducing SIM similarity in Task 3. OR and NOT gates may be easier for the LLM because they allow simpler Boolean decompositions and fewer deep nested transformations. However, better SIM evasion did not guarantee functional correctness.

### 4. Feedback Loop Impact
- How much did adding feedback (Task 4) improve over no feedback (Task 3)?
- What types of errors did the feedback loop catch most often?

  Task 4 did not improve over Task 3 in this run. Task 3 mapped 8/8 gate types with SIM 0.2500, while Task 4 mapped only 7/8 gate types with SIM 0.3300. The feedback loop increased the average trials to 3.5, but it did not fix functional correctness.

### 5. Practical Implications
- Which approach would you recommend for real IP piracy detection evasion?
- What are the trade-offs between different strategies?
- How might defenders counter these techniques?

  For real IP piracy detection evasion, I would not recommend relying only on live LLM rewriting because the outputs were not reliably functionally correct. The reference cached mapping was the only successful approach, showing that pre-validated transformations are much safer. Defenders should use more than text similarity; they should combine SIM detection with formal equivalence checking, graph-based circuit comparison, and structural analysis.

### Your Final Analysis:
```
TODO: Students write their analysis here
Best strategy overall:
The GPT3dot5 reference cached mapping was the best overall strategy. It achieved functional correctness, had a SIM score of 0.2500, and successfully evaded detection.

Key insights:
The live LLM-based methods could reduce similarity, but they failed to preserve functional correctness. Task 1 had the lowest SIM score at 0.2200, but its functional correctness was unknown due to syntax/parsing issues. Task 3 and Task 4 showed that divide-and-conquer mapping is more structured, but incorrect local mappings can still break the full circuit.

Surprising findings:
The reference GPT3dot5 cached mapping outperformed the live GPT-4 runs. This suggests that pre-validated mappings are more reliable than fresh LLM-generated outputs. It was also surprising that adding feedback in Task 4 did not improve results over Task 3; instead, Task 4 had a worse SIM score and mapped fewer gate types.

Recommendations:
LLM-generated circuit rewrites should always be verified using formal equivalence tools like Yosys. A low SIM score alone is not enough because the rewritten circuit may not work correctly. For defense, piracy detection should combine text similarity, structural graph comparison, and functional equivalence checking.
```